In [2]:
import tensorflow as tf
import numpy as np
import json
import pandas as pd
import os
from ipywidgets import interact
import ipywidgets as widgets

from Bio import SeqIO
from tqdm import tqdm
from pathlib import Path

2025-08-05 19:18:45.105795: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
def deserialize(serialized_example, metadata):

    feature_map = {
        'sequence': tf.io.FixedLenFeature([], tf.string),
        'target': tf.io.FixedLenFeature([], tf.string),
    }
    example = tf.io.parse_example(serialized_example, feature_map)
    sequence = tf.io.decode_raw(example['sequence'], tf.bool)
    sequence = tf.reshape(sequence, (metadata['seq_length'], 4))
    sequence = tf.cast(sequence, tf.float32)

    target = tf.io.decode_raw(example['target'], tf.float16)
    target = tf.reshape(target, (metadata['target_length'], metadata['num_targets']))
    target = tf.cast(target, tf.float32)

    return {'sequence': sequence, 'target': target}


def load_tfrecord_to_numpy(tfrecord_path, metadata):
    dataset = tf.data.TFRecordDataset([tfrecord_path], compression_type='ZLIB')
    dataset = dataset.map(lambda x: deserialize(x, metadata))
    sequences = []
    targets = []
    for example in dataset:
        sequences.append(example['sequence'].numpy())
        targets.append(example['target'].numpy())
    sequences = np.stack(sequences)
    targets = np.stack(targets)
    return {'sequence': sequences, 'target': targets}


def onehot_to_seq(arr):
    # arr: (131072, 4), valores float32
    indices = np.argmax(arr, axis=1)
    return ''.join(np.array(['a', 'c', 'g', 't'])[indices])


def get_hash(seq):
    return hashlib.sha256(seq.encode()).hexdigest()


def find_matches(genome_fasta, tfrecord_hashes, window_size=131072):
    for record in SeqIO.parse(genome_fasta, "fasta"):
        chrom = record.id
        seq = str(record.seq).upper()
        matches = []
        for i in range(len(seq) - window_size + 1):
            subseq = seq[i:i+window_size]
            h = get_hash(subseq)
            if h in tfrecord_hashes:
                matches.append((chrom, i, i+window_size, h))
                print(f"Match at {chrom}:{i}-{i+window_size}")
        return matches  # podés guardar o devolver esto


def get_all_hashes():
    hashes = [ open("sequence_hashes/"+x, "rt").readlines() for x in os.listdir("sequence_hashes") ]
    return hashes


def flatten_list(lst):
    return[x for y in lst for x in y]


def write_fasta(sequences, output_path):
    with open(output_path, "w") as f:
        for name, seq in sequences:
            f.write(f">{name}\n")
            # cortar en líneas de 80 caracteres
            for i in range(0, len(seq), 80):
                f.write(seq[i:i+80] + "\n")


def get_coords_from_blast(record_id):
    return pd.read_csv(f"alignments/{record_id}_blast_results_subset_top.tsv", sep='\t', header=None).set_axis(["seqID", "chromosome", "start", "end", "e_value", "bitscore"], axis=1)


def extract_region(region, expected_length=131_072):
    if not str(region.chromosome).startswith("chr"):
        chromosome = f"chr{region.chromosome}"
    else:
        chromosome = region.chromosome
    start = int(region.start)
    end   = int(region.end)
    sequence = mm10_per_chr[chromosome][start:end]

    assert (len(sequence)) == expected_length, f"Difference between start and end is not {expected_length} but {len(sequence)}."
    return sequence


def get_all_mouse_sequences(path="data/datasets/basenji/mouse/tfrecords/"):
    
    metadata = {
      'seq_length': 131072,
      'target_length': 896,
      'num_targets': 1643
    }
    all_seqs = []
    for basename in tqdm(os.listdir(path)):
        tfrecord_path = path + basename
        record_data = load_tfrecord_to_numpy(tfrecord_path, metadata)
        seqs = record_data['sequence'].astype(np.int8)
        all_seqs.append(seqs)
    
    all_seqs = np.concatenate(all_seqs, axis=0)
    return all_seqs


def get_hg38():
    genome_fasta = "data/datasets/ref/human/mm10.fa"
    seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }
    return seqs_per_chr


def get_mm10():
    genome_fasta = f"{MOUSE_REF_FOLDER}/mm10.fa"
    seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }
    return seqs_per_chr
    

def get_human_record_ids():
        
    return sorted(
        [ f.replace(".tfr", "") for f in os.listdir(HUMAN_TFR_FOLDER) ], key=lambda x: (x.split('-')[0], int(x.split('-')[2]))
    )


def get_mouse_record_ids():
    
    return sorted(
        [ f.replace(".tfr", "") for f in os.listdir(MOUSE_TFR_FOLDER) ], key=lambda x: (x.split('-')[0], int(x.split('-')[2]))
    )


def get_sequences_for_record(record_id):
    seq_for_record = { 
        record.id: str(record.seq).lower() 
        for record in tqdm(SeqIO.parse(f"fasta_seq/{record_id}.fasta", "fasta"))
    }    
    return seq_for_record


def get_human_basenji_regions():
    human_seqs = pd.read_csv("data/datasets/basenji/human/sequences.bed", sep='\t', header=None)
    human_seqs = human_seqs.set_axis(["chromosome", "start", "end", "subset"], axis=1)
    return human_seqs


def get_mouse_basenji_regions():
    mouse_seqs = pd.read_csv("data/datasets/basenji/mouse/sequences.bed", sep='\t', header=None)
    mouse_seqs = mouse_seqs.set_axis(["chromosome", "start", "end", "subset"], axis=1)
    return mouse_seqs


def expand_regions(regions_df, to_left:int=131_072, to_right:int=131_072):
    regions_df.start -= to_left
    regions_df.end += to_right
    return regions_df


def get_region_from_npy_filename(filename):
    return {
      'chr': filename.split("_")[0], 
      'start': int(filename.split("_")[1]), 
      'end': int(filename.split("_")[2].split(".")[0])
    }

In [ ]:
MOUSE_REF_FOLDER = "/home/rbonazzola/data/datasets/ref/mouse"
MOUSE_TFR_FOLDER = "/home/rbonazzola/data/datasets/basenji/mouse/tfrecords"
mm10_per_chr = get_mm10()

0it [00:00, ?it/s]

18it [00:10,  2.17it/s]

In [1]:
SEQLEN = 131072
TO_LEFT, TO_RIGHT = SEQLEN, SEQLEN
WHICH_RECORD = 5

record_id = (record_ids := get_mouse_record_ids())[WHICH_RECORD]

print(f"{record_id=}")
sequences_from_tfr = get_sequences_for_record(record_id)
blast_results_df   = get_coords_from_blast(record_id)

expanded_regions_df = expand_regions(blast_results_df, to_left=1+TO_LEFT, to_right=0+TO_RIGHT)
sequences_from_ref  = expanded_regions_df.apply(extract_region, axis=1, expected_length=SEQLEN+TO_LEFT+TO_RIGHT)

# if this gives True, we are good
assert all([ v == sequences_from_ref.iloc[i][TO_LEFT:-TO_RIGHT] for i, (k, v) in enumerate(sequences_from_tfr.items()) ])

NameError: name 'get_mouse_record_ids' is not defined

In [ ]:
mouse_seqs_df = get_mouse_basenji_regions()
mouse_seqs_set = mouse_seqs_df.apply(extract_region, axis=1).to_list())

0it [00:00, ?it/s]

66it [00:21,  3.09it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'data/datasets/basenji/mouse/sequences.bed'